# Apex Retail Intelligence — Phase 3: Bronze Layer

**Deliverable:** Bronze Layer Script

---

### What this notebook does

| Step | Action | Detail |
| --- | --- | --- |
| 1 | Read Landing Parquet | Historical + Incremental for each dataset |
| 2 | Write Bronze (Delta) | Inject `ingested_at` audit-trail column |
| 3 | Load strategy | Historical → **overwrite** (seed) · Incremental → **append** (no dedup) |

---

### Idempotency notes

* **Historical** — uses `overwrite` mode; safe to re-run for a full reseed.
* **Incremental** — uses `append` mode; re-running **will** duplicate rows at Bronze by design.
* Deduplication is intentionally deferred to Silver's MERGE step.



Cell 1 – Imports

In [0]:
from pyspark.sql.functions import current_timestamp, lit

2 – Paths

In [0]:
LANDING_BASE = "/Volumes/apex_retail/landing/parquet"
BRONZE_BASE = "/Volumes/apex_retail/landing/bronze"

# Process BOTH modes in sequence: Historical (overwrite/seed) then Incremental (append)
LOAD_MODES = ["Historical", "Incremental"]

DATASETS = ["customer", "product", "sales"]

3 – Create Bronze Folder

In [0]:
dbutils.fs.mkdirs(BRONZE_BASE)

True

4 – Bronze Processing

In [0]:
from pyspark.sql.functions import current_timestamp, lit

for LOAD_MODE in LOAD_MODES:
    print(f"\n{'='*60}")
    print(f"  PROCESSING: {LOAD_MODE.upper()} LOAD")
    print(f"{'='*60}")

    for dataset in DATASETS:
        source_path = f"{LANDING_BASE}/{LOAD_MODE.lower()}/{dataset}"
        bronze_path = f"{BRONZE_BASE}/{dataset}"

        print(f"\n  Reading: {source_path}")
        df = spark.read.parquet(source_path)

        bronze_df = (
            df
            .withColumn("ingested_at", current_timestamp())
            .withColumn("source_load_type", lit(LOAD_MODE))
        )

        if LOAD_MODE.lower() == "historical":
            bronze_df.write \
                .format("delta") \
                .mode("overwrite") \
                .save(bronze_path)
            print(f"  ✓ {dataset} — Historical loaded (overwrite) | {bronze_df.count()} rows")

        else:
            bronze_df.write \
                .format("delta") \
                .mode("append") \
                .save(bronze_path)
            print(f"  ✓ {dataset} — Incremental appended | {bronze_df.count()} rows")

print(f"\n{'='*60}")
print("  BRONZE LAYER COMPLETE")
print(f"{'='*60}")


  PROCESSING: HISTORICAL LOAD

  Reading: /Volumes/apex_retail/landing/parquet/historical/customer
  ✓ customer — Historical loaded (overwrite) | 50 rows

  Reading: /Volumes/apex_retail/landing/parquet/historical/product
  ✓ product — Historical loaded (overwrite) | 30 rows

  Reading: /Volumes/apex_retail/landing/parquet/historical/sales
  ✓ sales — Historical loaded (overwrite) | 100 rows

  PROCESSING: INCREMENTAL LOAD

  Reading: /Volumes/apex_retail/landing/parquet/incremental/customer
  ✓ customer — Incremental appended | 10 rows

  Reading: /Volumes/apex_retail/landing/parquet/incremental/product
  ✓ product — Incremental appended | 5 rows

  Reading: /Volumes/apex_retail/landing/parquet/incremental/sales
  ✓ sales — Incremental appended | 20 rows

  BRONZE LAYER COMPLETE


5 – Validation

In [0]:
print("\n" + "="*60)
print("  BRONZE LAYER VALIDATION")
print("="*60)

for dataset in DATASETS:
    bronze_path = f"{BRONZE_BASE}/{dataset}"
    df = spark.read.format("delta").load(bronze_path)

    # Count by load type
    hist_count = df.filter(df.source_load_type == "Historical").count()
    incr_count = df.filter(df.source_load_type == "Incremental").count()

    print(f"\n  {dataset.upper()}")
    print(f"    Total rows:       {df.count()}")
    print(f"    Historical rows:  {hist_count}")
    print(f"    Incremental rows: {incr_count}")
    print(f"    Columns:          {len(df.columns)}")
    print(f"    Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    display(df.limit(5))

print("\n" + "="*60)
print("  ✓ All Bronze tables contain Historical + Incremental data")
print("="*60)


  BRONZE LAYER VALIDATION

  CUSTOMER
    Total rows:       60
    Historical rows:  50
    Incremental rows: 10
    Columns:          10
    Schema: ['customer_id:StringType()', 'first_name:StringType()', 'last_name:StringType()', 'email:StringType()', 'phone:StringType()', 'city:StringType()', 'state:StringType()', 'signup_date:StringType()', 'ingested_at:TimestampType()', 'source_load_type:StringType()']


customer_id,first_name,last_name,email,phone,city,state,signup_date,ingested_at,source_load_type
C0001,Alice,Smith,cust1@example.com,555-1000,New York,NY,2023-01-01,2026-08-01T14:44:35.297Z,Historical
C0002,Bob,Jones,cust2@example.com,555-1001,Chicago,IL,2023-01-08,2026-08-01T14:44:35.297Z,Historical
C0003,Carol,Brown,cust3@example.com,555-1002,Houston,TX,2023-01-15,2026-08-01T14:44:35.297Z,Historical
C0004,Dave,Wilson,cust4@example.com,555-1003,Phoenix,AZ,2023-01-22,2026-08-01T14:44:35.297Z,Historical
C0005,Eve,Taylor,cust5@example.com,555-1004,Dallas,TX,2023-01-29,2026-08-01T14:44:35.297Z,Historical



  PRODUCT
    Total rows:       35
    Historical rows:  30
    Incremental rows: 5
    Columns:          7
    Schema: ['product_id:StringType()', 'product_name:StringType()', 'category:StringType()', 'price:StringType()', 'stock_quantity:StringType()', 'ingested_at:TimestampType()', 'source_load_type:StringType()']


product_id,product_name,category,price,stock_quantity,ingested_at,source_load_type
P0001,Product_B1,Electronics,15.5,110,2026-08-01T14:44:37.534Z,Historical
P0002,Product_C2,Clothing,21.0,120,2026-08-01T14:44:37.534Z,Historical
P0003,Product_D3,Home,26.5,130,2026-08-01T14:44:37.534Z,Historical
P0004,Product_E4,Sports,32.0,140,2026-08-01T14:44:37.534Z,Historical
P0005,Product_F5,Food,37.5,150,2026-08-01T14:44:37.534Z,Historical



  SALES
    Total rows:       120
    Historical rows:  100
    Incremental rows: 20
    Columns:          8
    Schema: ['sale_id:StringType()', 'customer_id:StringType()', 'product_id:StringType()', 'quantity:StringType()', 'total_amount:StringType()', 'sale_date:StringType()', 'ingested_at:TimestampType()', 'source_load_type:StringType()']


sale_id,customer_id,product_id,quantity,total_amount,sale_date,ingested_at,source_load_type
S00001,C0041,P0015,2,99.91,2023-01-15,2026-08-01T14:44:39.601Z,Historical
S00002,C0008,P0013,5,236.69,2023-01-18,2026-08-01T14:44:39.601Z,Historical
S00003,C0002,P0009,1,438.5,2023-01-21,2026-08-01T14:44:39.601Z,Historical
S00004,C0048,P0030,3,46.94,2023-01-24,2026-08-01T14:44:39.601Z,Historical
S00005,C0018,P0021,5,405.93,2023-01-27,2026-08-01T14:44:39.601Z,Historical



  ✓ All Bronze tables contain Historical + Incremental data
